# Training dataset statistics

Compute basic trajectory-level statistics for the tree-branches training dataset.

In [1]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../data/training/trajectories_tree_branches_5000_with_query.csv")
df = pd.read_csv(DATA_PATH)

# A query has multiple independent trajectories (branches). Each branch starts
# at step==1, so we split on every step reset within a query.
def extract_trajectory_lengths(group: pd.DataFrame) -> list[int]:
    lengths = []
    current = 0
    for _, row in group.iterrows():
        if row["step"] == 1 and current > 0:
            lengths.append(current)
            current = 0
        current += 1
    if current > 0:
        lengths.append(current)
    return lengths

all_lengths = []
for _, group in df.groupby("query_id", sort=False):
    all_lengths.extend(extract_trajectory_lengths(group))

lengths = pd.Series(all_lengths)
mean_length = lengths.mean()
std_length = lengths.std()

print(f"Queries: {df['query_id'].nunique()}")
print(f"Total transitions: {len(df)}")
print(f"Trajectories (branches): {len(lengths)}")
print(f"Mean trajectory length: {mean_length:.4f}")
print(f"Trajectory-length standard deviation: {std_length:.4f}")
print()
print("Trajectory length distribution:")
print(lengths.value_counts().sort_index())

Queries: 5000
Total transitions: 39967
Trajectories (branches): 19999
Mean trajectory length: 1.9984
Trajectory-length standard deviation: 0.7060

Trajectory length distribution:
1     5000
2    10030
3     4969
Name: count, dtype: int64


## Expert trajectories

The expert dataset stores one trajectory per query, so trajectory length is simply the number of rows per `query_id`.

In [2]:
EXPERT_PATH = Path("../data/training/trajectories_expert_5k.csv")
df_expert = pd.read_csv(EXPERT_PATH)

expert_lengths = df_expert.groupby("query_id").size()

print(f"Queries: {df_expert['query_id'].nunique()}")
print(f"Total transitions: {len(df_expert)}")
print(f"Mean trajectory length: {expert_lengths.mean():.4f}")
print(f"Trajectory-length standard deviation: {expert_lengths.std():.4f}")
print()
print("Trajectory length distribution:")
print(expert_lengths.value_counts().sort_index())

Queries: 5000
Total transitions: 8782
Mean trajectory length: 1.7564
Trajectory-length standard deviation: 0.9005

Trajectory length distribution:
1    2784
2     650
3    1566
Name: count, dtype: int64
